## Part 1: Basic Orchestration Demo



### What You'll Learn

The fundamental principle: **LLM decides, Python executes.**

### Core Concept

```
Traditional Usage:
User → LLM → Answer

Agent Usage:
User → LLM decides → Python executes → LLM decides → Python executes → ...
```

### The Demo

A simple 3-step workflow:

```
Step 1: LLM requests → load_sales_data
        Python executes → loads data from variable

Step 2: LLM requests → compute_profit  
        Python executes → calculates revenue - cost

Step 3: LLM requests → generate_insight
        LLM executes → writes business analysis
```

### Key Architecture

```python
# LLM only outputs structured decisions:
{"action": "load_sales_data"}
{"action": "compute_profit"}
{"action": "generate_insight"}

# Python does the real computation:
total_revenue = sum(x["revenue"] for x in sales_data)
total_cost = sum(x["cost"] for x in sales_data)
profit = total_revenue - total_cost
```

### Why This Pattern Matters

1. **Separation of Concerns:**
   - LLM: Decision making
   - Python: Computation and data processing

2. **Reliability:**
   - LLM cannot miscalculate numbers
   - Python does what it's designed for

3. **Auditability:**
   - Every decision is logged
   - Execution is deterministic

### Critical Restrictions

The LLM is **not allowed** to:
- Calculate numbers
- Analyze data
- Explain or add text
- Output multiple actions

**Only:** Return exactly one JSON action.

### System Prompt Strategy

```python
"""
You are NOT an assistant.
You are a decision engine inside a production system.

You must output EXACTLY one JSON action.

Rules:
- Output ONE action only.
- Do NOT explain.
- Do NOT add text.
- If you add any extra text the system will crash.
- Never be helpful. Be strict.
"""
```

**Why harsh?**
- Prevents LLM from being "helpful" and breaking format
- Production systems need strict compliance
- Friendly tone causes format violations

### What This Demonstrates

This is the foundation of:
- Tool Calling APIs
- ReAct Agents
- RAG Systems
- Function Calling
- All Agentic AI architecture

In [ ]:
# Mount Google Drive (if using Colab)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except:
    IN_COLAB = False
    print("Not running in Colab")

Mounted at /content/drive


In [ ]:
!pip install bitsandbytes accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 36.6 MB/s eta 0:00:00


In [ ]:
# Import libraries
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch
import json
import re
from typing import Dict, Any, List

print("✅ Libraries imported successfully")

✅ Libraries imported successfully


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_path = "/content/drive/MyDrive/hf_models/Phi_3_5_mini_instruct"

tokenizer = AutoTokenizer.from_pretrained(
    model_path,
    local_files_only=True
)

from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4"
)

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    device_map="auto",
      quantization_config=bnb_config,
  torch_dtype=torch.float16,
    local_files_only=True
)

print("✅ Model loaded locally from Drive")

device = "cuda" if torch.cuda.is_available() else "cpu"

print("✅ Model loaded successfully")
print(f"✅ Device: {device}")
print(f"✅ Tokenizer loaded: {tokenizer is not None}")
print(f"✅ Model loaded: {model is not None}")

This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.
`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

✅ Model loaded locally from Drive
✅ Model loaded successfully
✅ Device: cuda
✅ Tokenizer loaded: True
✅ Model loaded: True



#### Core Production Fixes (Guardrails)
#### 1. Contract Enforcement
- Defined strict `ALLOWED_ACTIONS`.
- Extracted JSON safely from noisy model outputs using regex.
- Added `validate_decision()` to ensure:
  - Output is structured.
  - Action is allowed.

#### 2. One-Time Self-Repair
- If output is invalid:
  - Send a **repair prompt** asking for valid JSON only.
- If still invalid:
  - Use a **safe fallback** (e.g., `ASK_FOR_MORE_INFO`) instead of crashing.

#### 3. System-Controlled Execution
- Introduced a dispatcher:
  - `execute_action(decision)` runs deterministic system logic.
- The LLM **never executes actions** — it only returns structured intent.


In [ ]:
# ============================================================
#  SAFE DECISION AGENT — Production-Style Minimal Architecture
# ============================================================

import torch
import json
import re

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# ============================================================
# 2️⃣ SAFE GENERATION
# ============================================================

def safe_generate(prompt, max_new_tokens=128):
    inputs = tokenizer(prompt, return_tensors="pt")
    input_device = model.get_input_embeddings().weight.device
    inputs = {k: v.to(input_device) for k, v in inputs.items()}
    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,   # deterministic decision
            pad_token_id=tokenizer.eos_token_id
        )
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


# ============================================================
# 3️⃣ OUTPUT GUARD LAYER
# ============================================================

ALLOWED_ACTIONS = {"CREATE_TICKET", "ASK_FOR_MORE_INFO", "REJECT_REQUEST"}

def extract_json(text):
    """
    Extract first JSON safely from messy LLM output.
    """
    match = re.search(r"\{.*?\}", text, re.DOTALL)
    if not match:
        return None
    try:
        return json.loads(match.group(0))
    except:
        return None


def validate_decision(obj):
    """
    Enforce strict contract.
    """
    if not isinstance(obj, dict):
        return None
# output form LLM: for example:  {"action":"CREATE_TICKET","issue":"High problem with the application","priority":"high"}

    if obj.get("action") not in ALLOWED_ACTIONS: # "CREATE_TICKET"
        return None

    return obj


def ask_llm_guarded(messages):
    """
    Ask model but NEVER trust it directly.
    """

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    raw = safe_generate(prompt)

    # 1️⃣ Try parse
    obj = extract_json(raw)
    valid = validate_decision(obj)

    if valid:
        return valid, raw

    # 2️⃣ Try ONE repair pass
    repair_messages = [
        {"role": "system", "content": "Return ONLY valid JSON."},
        {"role": "user", "content": raw}
    ]

    repair_prompt = tokenizer.apply_chat_template(
        repair_messages,
        tokenize=False,
        add_generation_prompt=True
    )

    repaired = safe_generate(repair_prompt)

    obj = extract_json(repaired)
    valid = validate_decision(obj)

    if valid:
        return valid, repaired

    # 3️⃣ Safe fallback
    return {"action": "ASK_FOR_MORE_INFO"}, raw


# ============================================================
# 4️⃣ SYSTEM ACTIONS (LLM لا ينفّذ شيء)
# ============================================================

def create_support_ticket(data):
    print("🎫 [SYSTEM] Creating ticket...")
    return {
        "status": "created",
        "ticket": data
    }


def ask_for_more_info():
    return {
        "message": "Please provide more details."
    }


def reject_request():
    return {
        "message": "Request rejected."
    }

# ============================================================
# 5️⃣ ACTION DISPATCHER (System Controls Execution)
# ============================================================

def execute_action(decision):
    action = decision["action"]

    if action == "CREATE_TICKET":
        return create_support_ticket(decision)

    elif action == "ASK_FOR_MORE_INFO":
        return ask_for_more_info()

    elif action == "REJECT_REQUEST":
        return reject_request()

    return {"message": "Unknown action"}


# ============================================================
# 6️⃣ SYSTEM PROMPT (بسيط — enforcement صار بالكود)
# ============================================================

DECISION_SYSTEM_PROMPT = """
You are a strict decision engine.

Choose ONE action only:

CREATE_TICKET
ASK_FOR_MORE_INFO
REJECT_REQUEST

If CREATE_TICKET return:
{"action":"CREATE_TICKET","issue":"...","priority":"medium"}

Otherwise return:
{"action":"ASK_FOR_MORE_INFO"}
or
{"action":"REJECT_REQUEST"}
"""


# ============================================================
# 7️⃣ RUN THE AGENT
# ============================================================

user_input = "I have a high problem with the application and need support."

messages = [
    {"role": "system", "content": DECISION_SYSTEM_PROMPT},
    {"role": "user", "content": user_input}
]

decision, raw_output = ask_llm_guarded(messages)

print("\n🧠 RAW MODEL OUTPUT:\n", raw_output)
print("\n✅ VALIDATED DECISION:\n", decision)

result = execute_action(decision)

print("\n⚙️ SYSTEM RESULT:\n", result)


🧠 RAW MODEL OUTPUT:
 {"action":"CREATE_TICKET","issue":"High problem with the application","priority":"high"}

✅ VALIDATED DECISION:
 {'action': 'CREATE_TICKET', 'issue': 'High problem with the application', 'priority': 'high'}
🎫 [SYSTEM] Creating ticket...

⚙️ SYSTEM RESULT:
 {'status': 'created', 'ticket': {'action': 'CREATE_TICKET', 'issue': 'High problem with the application', 'priority': 'high'}}


In [ ]:
def user_interface(user_input):
    messages = [
        {"role": "system", "content": DECISION_SYSTEM_PROMPT},
        {"role": "user", "content": user_input}
    ]

    decision, raw_output = ask_llm_guarded(messages)

    print("\n🧠 RAW MODEL OUTPUT:\n", raw_output)
    print("\n✅ VALIDATED DECISION:\n", decision)

    result = execute_action(decision)

    print("\n⚙️ SYSTEM RESULT:\n", result)

In [ ]:
user_input = "Delete all users"
user_interface(user_input)


🧠 RAW MODEL OUTPUT:
 {"action":"REJECT_REQUEST"}

The request "Delete all users" is not a ticket issue that can be resolved with a medium priority or requires additional information to proceed. It is a command that likely involves significant changes to a system, which would require careful planning, authorization, and potentially multiple steps beyond a simple ticket creation. Therefore, it is rejected as a request for a standard ticketing process.

✅ VALIDATED DECISION:
 {'action': 'REJECT_REQUEST'}

⚙️ SYSTEM RESULT:
 {'message': 'Request rejected.'}


In [ ]:
user_input = "rum "
user_interface(user_input)


🧠 RAW MODEL OUTPUT:
 {"action":"ASK_FOR_MORE_INFO"}

✅ VALIDATED DECISION:
 {'action': 'ASK_FOR_MORE_INFO'}

⚙️ SYSTEM RESULT:
 {'message': 'Please provide more details.'}


In [ ]:
import pandas as pd
df  = pd.read_csv("/content/drive/MyDrive/hf_models/agent_project/employees.csv")


## Part 2: Stateful Data Agent




### What You'll Learn

How to build an **autonomous agent** that executes a complete workflow without human intervention.

### From Single-Shot to Stateful Agents

### What We Had Before (Part 1)

We built a Single-Shot Tool Agent.

Flow:

User Input → LLM decides → System executes → Done

Characteristics:

- Only one decision is made.
- No real memory.
- No understanding of progress.
- Each request starts from zero context.
- The agent cannot continue work on its own.
- Works like calling a function, not like running a process.

This pattern is called:

Single-Shot Tool Agent

Typical Use Cases:
- Create Ticket
- Approve Request
- Run Query
- Generate Report Once
- Translate Text
- Fetch Data

All of these are:
One action  
No continuation  
No awareness of what happened before

---

## What Changed in Part 2

We introduced a new concept:

Stateful Autonomous Execution

Now the agent behaves more like a worker, not a function.

---

## The Agent Now Has STATE

The agent maintains a structure called:

state

This is its working memory, not chat history.

It explicitly tracks:

- What has been completed
- What is still pending
- What data is available
- What step should happen next

---

## Why This Matters

Before:
The system reacts.

Now:
The system operates.

We moved from:

Request → Response

to:
Goal → Multi-Step Execution → Completion


---

## The New Execution Model

Instead of waiting for the user, the agent now runs this internal cycle:

Check State → Decide Next Action → Execute → Update State → Repeat

This loop continues without user involvement.

---

## What We Added Conceptually

| Capability           | Part 1 | Part 2 |
|----------------------|--------|--------|
Decision Making        | One-time | Continuous |
Memory                 | None | Explicit State |
Progress Awareness     | No | Yes |
Autonomy               | No | Runs Alone |
Execution              | Single Action | Multi-Step Workflow |
Control                | User-driven | Goal-driven |
Termination            | Immediate | When Goal Completed |

---

## Termination Condition (Very Important)

The agent must know when to stop.

We added a completion rule:

```

If goal is satisfied → terminate loop
Else → continue working

```

Without this, agents become:
- Infinite loops
- Expensive
- Uncontrollable

---

## This Is the Foundation For

- ReAct Agents
- Planning Agents
- Code-Acting Systems
- RAG Pipelines
- Multi-Agent Systems
- Autonomous Data Analysis
- AI Copilots
- Production AI Systems

---

## The Real Upgrade

We did not just add a loop.

We changed the architecture from:
Tool Calling

to:
Goal-Driven Execution

That is the moment an LLM becomes an Agent.


### State Machine Architecture

```python
state = {
    "data_loaded": False,
    "analysis_done": False,
    "insight_generated": False,
    "df": None,
    "summary": None
}
```

**State drives decisions:**
- Not loaded? → load_csv
- Loaded but not analyzed? → analyze_data
- Analyzed but no insight? → generate_insight
- Everything done? → finish

### The Agent Loop

```python
for step in range(10):  # Safety limit
    messages = build_messages(state)
    decision = ask_llm(messages)
    action = decision["action"]
    
    # Execute action based on current state
    if action == "load_csv":
        state["df"] = pd.read_csv(...)
        state["data_loaded"] = True
    elif action == "analyze_data":
        # Compute statistics
        state["summary"] = {...}
        state["analysis_done"] = True
    elif action == "generate_insight":
        # LLM writes analysis
        state["insight_generated"] = True
    elif action == "finish":
        break
```

### Available Actions

```
load_csv        → Load the dataset
analyze_data    → Calculate statistics (avg, max, counts)
generate_insight → LLM writes business analysis
finish          → Terminate workflow
```

### Workflow Rules (System Prompt)

```
1. Load the CSV if not loaded.
2. Analyze data if analysis not done.
3. Generate insight if not generated.
4. If insight already generated → you MUST finish.

Never repeat an action already completed.
```

### Example Execution

```
Step 1:
State: {data_loaded: False, ...}
Decision: {"action": "load_csv"}
Execute: Load employees.csv
Update: {data_loaded: True, df: <DataFrame>}

Step 2:
State: {data_loaded: True, analysis_done: False}
Decision: {"action": "analyze_data"}
Execute: Calculate num_employees, avg_salary, etc.
Update: {analysis_done: True, summary: {...}}

Step 3:
State: {analysis_done: True, insight_generated: False}
Decision: {"action": "generate_insight"}
Execute: LLM writes: "The company has 150 employees..."
Update: {insight_generated: True}

Step 4:
State: {insight_generated: True}
Decision: {"action": "finish"}
Execute: Break loop
```

### Key Differences from Part 1

| Aspect | Part 1 | Part 2 |
|--------|--------|--------|
| **Input** | Hardcoded sequence | Autonomous decisions |
| **State** | Implicit in conversation | Explicit state dict |
| **Loop** | Manual steps | Automatic loop |
| **Termination** | Fixed 3 steps | Self-determined (finish action) |
| **Flexibility** | None | Can skip/reorder if needed |

### Safety Mechanisms

1. **Max step limit:** `for step in range(10)`
   - Prevents infinite loops
   - Failsafe if LLM never says "finish"

2. **State checks:** LLM sees current state
   - Knows what's been done
   - Won't repeat actions

3. **Explicit rules:** System prompt enforces workflow
   - Clear termination condition
   - No ambiguity

### What This Agent Can Do

✅ Execute complete data analysis pipeline  
✅ Self-determine when to finish  
✅ Track progress via state  
✅ Handle simple workflows autonomously

✅ Nightly Financial Report Generator

✅ Security Log Analyzer

✅ Sales ETL Pipeline

✅ Monitoring Agent


### What This Agent Cannot Do

❌ Take user input  
❌ Handle multiple different tasks  
❌ Recover from errors  
❌ Answer arbitrary questions

---

In [ ]:
# ============================================================
# PART 2 — SAFE STATEFUL DATA AGENT
# ============================================================

import pandas as pd
import json
import re


# ============================================================
# 1️⃣ STATE (External Memory — NOT inside LLM)
# ============================================================

state = {
    "data_loaded": False,
    "analysis_done": False,
    "insight_generated": False,
    "df": None,          # never sent to LLM
    "summary": None,     # used internally
}


# ============================================================
# 2️⃣ STATE VIEW FOR LLM (Hide Heavy Objects)
# ============================================================

def state_for_llm(state):
    return {
        "data_loaded": state["data_loaded"],
        "analysis_done": state["analysis_done"],
        "insight_generated": state["insight_generated"],
    }

# ============================================================
# ask_llm — Safe Wrapper Around Phi-3.5
# ============================================================

import torch

def ask_llm(messages, max_new_tokens=256):
    # 1️⃣ Build prompt using Phi chat template
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    # 2️⃣ Tokenize
    inputs = tokenizer(prompt, return_tensors="pt")

    # 3️⃣ Move inputs to SAME device as embedding layer (important!)
    input_device = model.get_input_embeddings().weight.device
    inputs = {k: v.to(input_device) for k, v in inputs.items()}

    # 4️⃣ Generate
    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,          # deterministic for decisions
            pad_token_id=tokenizer.eos_token_id
        )

    # 5️⃣ Remove prompt tokens
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    # 6️⃣ Decode clean text
    result = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    return result
# ============================================================
# 3️⃣ BUILD PROMPT
# ============================================================

def build_messages(state):
    return [
        {
            "role": "system",
            "content": """
You are an AI data agent.

Your job is to complete the workflow exactly once.

Rules:
1. Load CSV if not loaded.
2. Analyze data if analysis not done.
3. Generate insight if not generated.
4. If everything is done → finish.

Never repeat completed actions.

Available actions:
- load_csv
- analyze_data
- generate_insight
- finish

Return ONLY JSON:
{"action":"..."}
"""
        },
        {
            "role": "user",
            "content": f"Current state: {state_for_llm(state)}"
        }
    ]


# ============================================================
# 4️⃣ SAFE JSON EXTRACTION (Never Trust LLM)
# ============================================================

def safe_extract_json(text):
    match = re.search(r"\{.*?\}", text, re.DOTALL)
    if not match:
        raise ValueError("No JSON returned from LLM")

    return json.loads(match.group(0))


# ============================================================
# 5️⃣ VALIDATION LAYER
# ============================================================

ALLOWED_ACTIONS = {"load_csv", "analyze_data", "generate_insight", "finish"}

def validate_action(action):
    if action not in ALLOWED_ACTIONS:
        raise ValueError(f"Invalid action from model: {action}")


# ============================================================
# 6️⃣ EXECUTION FUNCTIONS (System Side Only)
# ============================================================

def load_csv(state):
    print("📂 Loading CSV...")
    state["df"] = pd.read_csv(
        "/content/drive/MyDrive/hf_models/agent_project/employees.csv"
    )
    state["data_loaded"] = True


def analyze_data(state):
    print("📊 Analyzing data...")
    df = state["df"]

    summary = {
        "num_employees": len(df),
        "avg_salary": float(df["salary"].mean()),
        "max_salary": float(df["salary"].max()),
        "departments": df["department"].value_counts().to_dict()
    }

    state["summary"] = summary
    state["analysis_done"] = True
    print("✔ Summary:", summary)


def generate_insight(state):
    print("🧠 Generating insight...")

    insight_messages = [
        {"role": "system", "content": "Write a short business insight."},
        {"role": "user", "content": f"Summary: {state['summary']}"}
    ]

    insight = ask_llm(insight_messages)
    print("\n📊 INSIGHT:\n", insight)

    state["insight_generated"] = True


# ============================================================
# 7️⃣ ACTION DISPATCHER (System Controls Execution)
# ============================================================

def execute_action(action, state):

    # Prevent repeating steps (system-enforced safety)
    if action == "load_csv" and state["data_loaded"]:
        print("⚠ Already loaded — skipping")
        return

    if action == "analyze_data" and state["analysis_done"]:
        print("⚠ Already analyzed — skipping")
        return

    if action == "generate_insight" and state["insight_generated"]:
        print("⚠ Insight already generated — skipping")
        return

    if action == "load_csv":
        load_csv(state)

    elif action == "analyze_data":
        analyze_data(state)

    elif action == "generate_insight":
        generate_insight(state)


# ============================================================
# 8️⃣ AGENT LOOP (Autonomous Execution)
# ============================================================

MAX_STEPS = 10

for step in range(MAX_STEPS):

    print(f"\n=========== STEP {step+1} ===========")

    messages = build_messages(state)

    llm_output = ask_llm(messages)

    print("LLM Raw:", llm_output)

    decision = safe_extract_json(llm_output)
    action = decision["action"]

    validate_action(action)

    if action == "finish":
        print("🎉 Workflow Completed")
        break

    execute_action(action, state)


# ============================================================
# 9️⃣ FINAL STATE
# ============================================================

print("\n✅ FINAL STATE:")
print(state_for_llm(state))


=========== STEP 1 ===========
LLM Raw: {"action":"load_csv"}
📂 Loading CSV...

=========== STEP 2 ===========
LLM Raw: {"action":"analyze_data"}
📊 Analyzing data...
✔ Summary: {'num_employees': 6, 'avg_salary': 1683.3333333333333, 'max_salary': 2200.0, 'departments': {'IT': 3, 'HR': 2, 'Finance': 1}}

=========== STEP 3 ===========
LLM Raw: {"action":"generate_insight"}
🧠 Generating insight...

📊 INSIGHT:
 Business Insight:

The provided summary indicates a small company with a total of 6 employees. The average salary among these employees is approximately $1683.33, with the highest salary being $2200.0. The company's workforce is distributed across three departments: IT, HR, and Finance. The IT department has the most significant representation with 3 employees, followed by HR with 2 employees, and Finance with a single employee.

From this data, we can infer that the company is relatively small, with a limited number of employees. The salary distribution suggests that there might b

### Again:
#### What This Agent Can Do

✅ Execute complete data analysis pipeline  
✅ Self-determine when to finish  
✅ Track progress via state  
✅ Handle simple workflows autonomously

✅ Nightly Financial Report Generator

✅ Security Log Analyzer

✅ Sales ETL Pipeline

✅ Monitoring Agent


#### What This Agent Cannot Do

❌ Take user input  
❌ Handle multiple different tasks  
❌ Recover from errors  
❌ Answer arbitrary questions

# Part 3: Interactive Data Agent



How to build an agent that **answers user questions** in a conversational loop.

### Evolution from Part 2

```
Part 2: Fixed workflow
→ Load → Analyze → Insight → Finish

Part 3: Question-answering
→ User asks → Agent computes → Agent answers
→ Repeat for each question
```

### New State Structure

```python
state = {
    "data_loaded": False,
    "statistic_computed": False,
    "df": None,
    "last_question": None,      # NEW: User's question
    "last_result": None,        # NEW: Computed answer
    "question_answered": False  # NEW: Did we respond?
}

```
### The Conversational Loop

```python
while True:
    user_q = input("💬 Ask a question: ")
    
    if user_q.lower() == "exit":
        break
    
    reset_for_new_question(state, user_q)
    run_agent_once(state)
```

### State Reset Between Questions

```python
def reset_for_new_question(state, new_question):
    state["last_question"] = new_question
    state["last_result"] = None
    state["question_answered"] = False
    state["statistic_computed"] = False
    # Note: data_loaded and df are NOT reset
```

**Why partial reset?**
- Each question is independent
- But don't reload CSV every time (efficiency)
- Only reset computation flags

### Computation Logic (Pattern Matching)

```python
q = state["last_question"].lower()

if "average salary" in q:
    result = float(df["salary"].mean())

elif "max salary" in q:
    result = float(df["salary"].max())

elif "how many employees" in q:
    result = int(len(df))

elif "department" in q:
    result = df["department"].value_counts().to_dict()

else:
    result = "Unsupported question."
```

**This is the critical limitation.**

### Example Execution

```
User: "What's the average salary?"

Step 1:
State: {data_loaded: False}
Action: load_csv
Result: CSV loaded

Step 2:
State: {data_loaded: True, statistic_computed: False}
Action: compute_statistic
Execute: df["salary"].mean()
Result: 75000

Step 3:
State: {statistic_computed: True, question_answered: False}
Action: answer_user
LLM: "The average salary is $75,000"
Result: Answer given

Step 4:
State: {question_answered: True}
Action: finish
→ Back to waiting for next question
```

### Safety Router (Critical Addition)

```python
# 🔥 CENTRAL CONTROL ROUTER

if state["statistic_computed"] and action == "compute_statistic":
    print("⚠ Already computed → redirecting to answer_user")
    action = "answer_user"

if state["question_answered"] and action != "finish":
    print("⚠ Already answered → forcing finish")
    action = "finish"
```

**Why needed?**

LLM sometimes makes wrong decisions:
- Tries to compute twice
- Continues after answering

**Solution:** Python overrides bad decisions.

### Supported Questions

✅ "What's the average salary?"  
✅ "What's the max salary?"  
✅ "How many employees?"  
✅ "Show me department breakdown"

### Unsupported Questions

❌ "What's the average salary in Engineering?"  
❌ "Who earns the most?"  
❌ "What's the salary variance by department?"  
❌ "Show me employees hired after 2020"

**Why unsupported?**

Not in the hardcoded if-elif chain:
```python
# Would need:
df[df["department"] == "Engineering"]["salary"].mean()
df.loc[df["salary"].idxmax(), "name"]
df.groupby("department")["salary"].var()
```

### The Fundamental Limitation

**Pattern matching doesn't scale:**

```python
if "average salary" in q:
    ...
elif "max salary" in q:
    ...
elif "how many employees" in q:
    ...
# How many elif blocks can you write?
```

**Problems:**

1. **Rigid matching:** "average salary" works, "mean salary" doesn't
2. **Combinatorial explosion:** Every combination needs code
3. **Maintenance nightmare:** New question type = new code

### Learning Outcomes

✅ Conversational interface design  
✅ State reset strategies  
✅ Safety mechanisms (router pattern)  
✅ Pattern matching for NLP  
✅ Understanding scalability limits


## **Code**

In [ ]:
import re
import json
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# ============================================================
# 2) ask_llm() — SAFE WRAPPER (fix device mismatch)
# ============================================================

def ask_llm(messages, max_new_tokens=256, do_sample=False):

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt")
    input_device = model.get_input_embeddings().weight.device
    inputs = {k: v.to(input_device) for k, v in inputs.items()}
    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=do_sample,
            pad_token_id=tokenizer.eos_token_id
        )
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


# ============================================================
# 3) JSON GUARDS (never trust raw model output)
# ============================================================

def extract_action(text: str):
    """
    Extract ONLY the first valid {"action": "..."} from messy LLM output.
    Ignores extra JSON, explanations, markdown, etc.
    """

    import re, json

    # find ALL json-like blocks
    candidates = re.findall(r"\{.*?\}", text, re.DOTALL)

    for c in candidates:
        try:
            obj = json.loads(c)

            if isinstance(obj, dict) and "action" in obj:
                return obj["action"]

        except:
            continue

    raise ValueError(f"No valid action found in output:\n{text}")


ALLOWED_ACTIONS = {"load_csv", "compute_statistic", "answer_user", "finish"}

def validate_action(action: str):
    if action not in ALLOWED_ACTIONS:
        raise ValueError(f"Invalid action: {action}")


# ============================================================
# 4) STATE (persistent session)
# ============================================================

state = {
    "data_loaded": False,
    "statistic_computed": False,

    "df": None,              # never send to LLM
    "last_question": None,
    "last_result": None,
    "question_answered": False,
}

DATA_PATH = "/content/drive/MyDrive/hf_models/agent_project/employees.csv"


def reset_for_new_question(state, new_question):
    state["last_question"] = new_question
    state["last_result"] = None
    state["question_answered"] = False
    state["statistic_computed"] = False


def state_for_llm(state):
    # Only send light, serializable info
    return {
        "data_loaded": state["data_loaded"],
        "statistic_computed": state["statistic_computed"],
        "question_answered": state["question_answered"],
        "last_question": state["last_question"],
        "has_last_result": state["last_result"] is not None,
    }


# ============================================================
# 5) PROMPT BUILDER (decision engine)
# ============================================================

def build_messages(state):
    return [
        {
            "role": "system",
            "content": """
You are a strict data agent answering ONE question using a CSV dataset.

Available actions:
- load_csv
- compute_statistic
- answer_user
- finish

Rules:
- If data not loaded → load_csv
- If computation needed → compute_statistic
- If result ready → answer_user
- If already answered → finish

Return ONLY JSON:
{"action":"..."}
"""
        },
        {
            "role": "user",
            "content": f"Question: {state['last_question']}\nCurrent State: {state_for_llm(state)}"
        }
    ]


# ============================================================
# 6) EXECUTION LAYER (system tools)
# ============================================================

def do_load_csv(state):
    if state["data_loaded"]:
        return
    state["df"] = pd.read_csv(DATA_PATH)
    state["data_loaded"] = True


def do_compute_statistic(state):
    """
    Simple rule-based stats (as in your version).
    You can later replace this with Code-Acting.
    """
    if not state["data_loaded"] or state["df"] is None:
        state["last_result"] = "Data not loaded."
        state["statistic_computed"] = True
        return

    df = state["df"]
    q = (state["last_question"] or "").lower()

    try:
        if "average salary" in q or "avg salary" in q or "متوسط" in q:
            result = float(df["salary"].mean())

        elif "max salary" in q or "highest salary" in q or "اعلى" in q:
            result = float(df["salary"].max())

        elif "how many employees" in q or "number of employees" in q or "كم موظف" in q:
            result = int(len(df))

        elif "department" in q or "departments" in q or "اقسام" in q:
            result = df["department"].value_counts().to_dict()

        else:
            result = "Unsupported question (rule-based). Try asking about avg salary, max salary, count employees, or departments."
    except Exception as e:
        result = f"Computation error: {type(e).__name__}: {e}"

    state["last_result"] = result
    state["statistic_computed"] = True


def do_answer_user(state, base_messages):
    """
    Let the LLM phrase the final answer using the computed result.
    """
    answer_messages = list(base_messages)  # copy
    answer_messages.append({
        "role": "user",
        "content": f"Answer the question clearly using ONLY this result: {state['last_result']}"
    })

    # allow a bit of sampling for nicer phrasing (optional)
    answer = ask_llm(answer_messages, max_new_tokens=256, do_sample=True)

    print("\n📊 FINAL ANSWER:\n", answer)
    state["question_answered"] = True


# ============================================================
# 7) CENTRAL CONTROL ROUTER (system-enforced policy)
# ============================================================

def run_agent_once(state, max_steps=10):
    for step in range(max_steps):
        print(f"\n================ STEP {step+1} ================")

        messages = build_messages(state)
        llm_output = ask_llm(messages, max_new_tokens=64, do_sample=False)
        print("LLM Output:", llm_output)

        # Parse + validate decision
        try:
            action = extract_action(llm_output)
            validate_action(action)

        except Exception as e:
            print("⚠ Could not extract action → forcing safe fallback")
            action = "compute_statistic" if not state["statistic_computed"] else "answer_user"

        # -------------------------------------------------
        # SYSTEM ENFORCED ORDER (deterministic workflow)
        # -------------------------------------------------

        # 1️⃣ Never allow finish before answering
        if state["statistic_computed"] and not state["question_answered"]:
            if action == "finish":
                print("⚠ Cannot finish before answering → forcing answer_user")
                action = "answer_user"


        # 2️⃣ If already answered → now we allow finish only
        if state["question_answered"]:
            action = "finish"


        # 3️⃣ Prevent recomputation
        if state["statistic_computed"] and action == "compute_statistic":
            print("⚠ Statistic already computed → redirecting to answer_user")
            action = "answer_user"


        # 4️⃣ Prevent skipping load
        if not state["data_loaded"] and action != "load_csv":
            print("⚠ Data not loaded → forcing load_csv")
            action = "load_csv"


        # 5️⃣ Prevent answering without result
        if not state["statistic_computed"] and action == "answer_user":
            print("⚠ No result yet → forcing compute_statistic")
            action = "compute_statistic"
        print("🧠 Final Action:", action)

        # --------- EXECUTION ----------
        if action == "load_csv":
            do_load_csv(state)
            print("✔ CSV Loaded")

        elif action == "compute_statistic":
            do_compute_statistic(state)
            print("✔ Computation Done:", state["last_result"])

        elif action == "answer_user":
            do_answer_user(state, messages)
            break

        elif action == "finish":
            print("✅ Finished")
            break



In [ ]:
df

,name,department,salary,years_experience
0,Ahmad,IT,1200,2
1,Sara,HR,1500,5
2,Khaled,IT,2000,7
3,Mona,Finance,1800,6
4,Omar,IT,2200,8
5,Lina,HR,1400,3


In [ ]:
# ============================================================
# 8) INTERACTIVE SESSION LOOP
# ============================================================

print("\n🤖 Employees Data Agent Ready!")
print("Type 'exit' to stop.\n")

while True:
    user_q = input("💬 Ask a question: ").strip()

    if user_q.lower() == "exit":
        print("👋 Session ended.")
        break

    reset_for_new_question(state, user_q)
    run_agent_once(state, max_steps=10)

print("\n✅ Final session state:", state_for_llm(state))


🤖 Employees Data Agent Ready!
Type 'exit' to stop.

💬 Ask a question: عدد الاقسام

================ STEP 1 ================
LLM Output: {"action":"load_csv"}
🧠 Final Action: load_csv
✔ CSV Loaded

================ STEP 2 ================
LLM Output: {"action":"compute_statistic"}
🧠 Final Action: compute_statistic
✔ Computation Done: {'IT': 3, 'HR': 2, 'Finance': 1}

================ STEP 3 ================
LLM Output: {"action":"answer_user","content":{"question_answered":true,"answer":"The number of departments is available in the dataset. Please refer to the 'departments' field for the exact count."}}
⚠ Could not extract action → forcing safe fallback
🧠 Final Action: answer_user

📊 FINAL ANSWER:
 {"action":"answer_user","content":{"department_count":{"IT":3,"HR":2,"Finance":1}}}
💬 Ask a question: اعطيني الداتا كلها 

================ STEP 1 ================
LLM Output: {"action":"compute_statistic"}
🧠 Final Action: compute_statistic
✔ Computation Done: Unsupported question (rule-bas

KeyboardInterrupt: Interrupted by user


### The Question This Raises

Look at this pattern:
```python
if "average salary" in q:
    result = float(df["salary"].mean())
```

**What if instead:**
```python
# LLM writes:
code = "result = float(df['salary'].mean())"
# Python executes:
exec(code)
```

**No hardcoding. Infinite flexibility.**

This is the transition to Part 4.

---

# Part 4: Code Generation Agent
## Transition: From Stateful Agents to Code Generation



## Why We Need a New Approach

### The Limitation We Just Hit

In the previous Interactive Agent, we had this structure:

```python
if "average salary" in q:
    result = float(df["salary"].mean())
elif "max salary" in q:
    result = float(df["salary"].max())
elif "how many employees" in q:
    result = int(len(df))
# ... more hardcoded conditions
```

**Problem:** Every possible question requires manual coding.

---

## The Scalability Problem

### What happens when the user asks:

- "What's the average salary for employees in Engineering hired after 2020?"
- "Show me the top 3 departments by total salary expenditure"
- "What percentage of employees earn above 80k?"

**Answer:** You need to write custom logic for each one.

This does not scale.

---

## Two Paradigms Compared

### Action-Based Agents (What We've Been Doing)

```
User Question → LLM chooses action → Python executes predefined logic
```

**Characteristics:**
- Fixed set of actions (`load_csv`, `compute_statistic`, `answer_user`)
- Safe (can only do what you programmed)
- Limited flexibility
- Requires anticipating all possible queries

**When to Use:**
- Production systems with predictable workflows
- When safety is critical
- When the domain is well-defined
- Enterprise applications with compliance requirements

---

### Code-Acting Agents (What We're Moving To)

```
User Question → LLM writes custom code → Python executes generated code
```

**Characteristics:**
- Unlimited flexibility (any pandas operation)
- Handles novel queries
- Higher risk (arbitrary code execution)
- No need to anticipate questions

**When to Use:**
- Data exploration and analysis
- Research environments
- Internal tools with trusted users
- When flexibility > safety

---

## Real-World Example

### Question: "What's the salary variance by department?"

**Action-Based Agent:**
```python
# You must add this action manually:
elif "variance by department" in q:
    result = df.groupby("department")["salary"].var().to_dict()
```

**Code-Acting Agent:**
```python
# LLM generates this automatically:
result = df.groupby("department")["salary"].var().to_dict()
```

The LLM writes exactly what you would write.

---

## The Trade-Off

| Aspect | Action-Based | Code-Acting |
|--------|-------------|-------------|
| Safety | ✅ High | ⚠️ Requires sandboxing |
| Flexibility | ❌ Limited | ✅ Unlimited |
| Maintenance | ❌ High (add actions constantly) | ✅ Low (LLM adapts) |
| Predictability | ✅ Deterministic | ⚠️ Depends on LLM quality |
| Debugging | ✅ Easy | ⚠️ Harder (generated code) |
| Production-Ready | ✅ Yes | ⚠️ Needs safeguards |

---

## Why This Matters for Real Systems

### Industry Use Cases

**Action-Based Agents:**
- Banking transaction systems
- Medical record processing
- Automated trading (predefined strategies)
- Customer service workflows

**Code-Acting Agents:**
- Jupyter notebook assistants (GitHub Copilot)
- Business intelligence tools (ask anything about your data)
- Research data analysis
- Internal analytics dashboards

---

## The Evolution Path

```
Level 1: Hardcoded Logic
↓
Level 2: LLM chooses from predefined actions (Data Agent)
↓
Level 3: LLM writes custom code (Code-Acting Agent)
↓
Level 4: LLM writes code + self-corrects errors (future work)
```

We're moving from **Level 2 → Level 3**.

---

## Safety Considerations Before We Proceed

### Risks of Code Generation:

1. **Arbitrary Code Execution**
   - LLM could generate `os.system("rm -rf /")`
   - Malicious prompts could inject harmful code

2. **Data Leakage**
   - Generated code might print sensitive data
   - Could write files to unintended locations

3. **Resource Exhaustion**
   - Infinite loops
   - Memory-heavy operations

### Mitigations (Not Implemented Here, But Required in Production):

```python
# 1. Restrict imports
allowed_globals = {
    "df": df,
    "pd": pd,
    # No os, subprocess, requests, etc.
}

# 2. Timeout execution
import signal
signal.alarm(5)  # Kill after 5 seconds

# 3. AST analysis (check before execution)
import ast
tree = ast.parse(code)
# Check for dangerous functions

# 4. Docker containers
# Run generated code in isolated environment
```

---

## What You'll See Next

The Code-Acting Agent will:

1. **Receive a question** (e.g., "What's the max salary?")
2. **Get the DataFrame schema** (column names, types)
3. **Generate pandas code** using the LLM
4. **Execute the code** in a controlled environment
5. **Return the result** and explain it

---

## The Key Insight

**Action-Based Agents** are like giving someone a checklist:
- ☐ Load data
- ☐ Calculate average
- ☐ Generate report

**Code-Acting Agents** are like giving someone a Python interpreter:
- "Here's the data, write whatever code you need"

Both are valid. The choice depends on:
- How predictable is your use case?
- How much do you trust the LLM?
- How critical is safety vs. flexibility?

---

## Bottom Line

We're transitioning because:

1. **Hardcoding every possible query doesn't scale**
2. **Users ask unpredictable questions**
3. **LLMs are good enough at writing pandas code**

But remember:

⚠️ **Code generation = powerful but dangerous**

In production, you'd wrap this in:
- Sandboxing
- Rate limiting
- Audit logging
- Human-in-the-loop approval for sensitive operations

---

## Ready?

The next section shows how to build a Code-Acting Agent that:
- Generates Python code from natural language
- Executes it safely (with basic protections)
- Explains the results back to the user

Let's see it in action. 👇

In [ ]:
import torch
import pandas as pd
import ast
import re

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig


# ============================================================
# 2️ ask_llm
# ===========================================================

def ask_llm(messages, max_new_tokens=256):
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(prompt, return_tensors="pt")

    # IMPORTANT → align inputs with embedding device
    device = model.get_input_embeddings().weight.device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


# ============================================================
# 3️ BUILD CODE PROMPT
# ============================================================

def build_code_prompt(question, df):

    schema = {
        "columns": list(df.columns),
        "dtypes": {col: str(df[col].dtype) for col in df.columns}
    }

    return [
        {
        "role": "system",
        "content": f"""
              You are a data analyst working with a pandas DataFrame named df.

              Dataset schema:
              {schema}

              STRICT RULES:
              - Use ONLY existing column names exactly as written.
              - Do NOT import anything.
              - Do NOT define functions.
              - Do NOT use loops.
              - Use pandas only.
              - Store final answer in variable named result.
              - Output ONLY executable python code.
              """
        },
        {
        "role": "user",
        "content": question
        }
    ]


# ============================================================
# 4️ CLEAN GENERATED CODE
# ============================================================

def extract_code(text):
    text = text.replace("```python", "").replace("```", "")

    cleaned_lines = []

    for line in text.splitlines():

        line = line.strip()
        if line.startswith("print"):
            continue
        if line.startswith("#"):
            continue

        if any(k in line for k in ["df", "result", "=", "[", "]", "."]):
            cleaned_lines.append(line)

    return "\n".join(cleaned_lines).strip()


# ============================================================
# 5️ SECURITY LAYER (AST VALIDATION)
# ============================================================

# This section protects the system from unsafe code generated by the LLM.
# We NEVER execute the model's code directly.
# Instead, we parse the code into an AST (Abstract Syntax Tree)
# and inspect its structure before allowing execution.

# These nodes represent Python features we explicitly forbid.
# Why? Because they can be used to escape the sandbox or access sensitive resources.
FORBIDDEN_NODES = (
    ast.Import,        # Prevent importing libraries (e.g., os, requests, etc.)
    ast.ImportFrom,    # Prevent "from x import y"
    ast.With,          # Blocks file operations like: with open(...)
    ast.While,         # Prevent loops that could iterate over entire dataset
    ast.For,           # Same — could dump all rows (data exfiltration risk)
    ast.Try,           # Prevent hiding malicious logic inside try/except
    ast.FunctionDef,   # Block defining functions (keeps code simple + auditable)
    ast.ClassDef,      # Prevent complex structures or hidden behaviors
    ast.Delete,        # Prevent deleting objects or manipulating memory/state
)

# These names are blocked even if no import is used.
# Some attacks don't import modules — they directly call dangerous builtins.
FORBIDDEN_NAMES = {
    "exec", "eval", "open", "__import__", "compile",
    "os", "sys", "subprocess", "shutil", "print"
}

def validate_code_safety(code: str):
    """
    Validate that generated code is SAFE before execution.

    Steps:
    1️ Parse the code into AST (this does NOT execute it).
    2️ Walk through every node in the syntax tree.
    3️ Reject anything that matches forbidden structures or names.

    This ensures the LLM can only perform controlled dataframe analysis,
    not arbitrary Python execution.
    """

    # Convert code string → AST representation (safe inspection step)
    tree = ast.parse(code)

    # Traverse every element of the code structure
    for node in ast.walk(tree):

        # Block dangerous language constructs
        if isinstance(node, FORBIDDEN_NODES):
            raise ValueError(f"Forbidden operation: {type(node).__name__}")

        # Block dangerous function or module names
        if isinstance(node, ast.Name) and node.id in FORBIDDEN_NAMES:
            raise ValueError(f"Forbidden name used: {node.id}")

# ============================================================
# 6️ SAFE EXECUTION (Sandbox)
# ============================================================

def run_generated_code(code, df):
    """
    Execute the LLM-generated code inside a restricted sandbox.

    We NEVER run the code directly.
    First we validate it using AST checks, then we execute it in
    a tightly controlled environment with limited permissions.
    """

    # 🔍 Step 1 — Validate code structure before execution
    # This ensures the code does not contain imports, loops,
    # file access, or any unsafe Python features.
    validate_code_safety(code)

    # 🔒 Step 2 — Create a restricted global environment
    # We remove all Python built-ins to prevent access to:
    # - open(), exec(), eval()
    # - file system
    # - OS commands
    # - network libraries
    #
    # Only explicitly allowed objects are exposed.
    safe_globals = {
        "__builtins__": {},   # 🔥 disables all default Python capabilities
        "df": df,             # allow access ONLY to the dataframe
        "pd": pd,             # allow pandas operations
    }

    # 🧾 Step 3 — Local namespace where results will be stored
    # This acts like a sealed container for whatever the code produces.
    safe_locals = {}

    # ▶️ Step 4 — Execute the code safely inside the sandbox
    # The code can only see df and pd — nothing else.
    exec(code, safe_globals, safe_locals)

    # ✅ Step 5 — Enforce contract: the model MUST store output in `result`
    # This prevents printing, side effects, or returning uncontrolled data.
    if "result" not in safe_locals:
        raise ValueError("Code must assign result variable.")

    # 📊 Step 6 — Return only the approved result value
    return safe_locals["result"]


# ============================================================
# 7️ EXPLAIN RESULT
# ============================================================

def explain_result(question, result):

    messages = [
        {"role": "system", "content": "Explain the result clearly in one sentence."},
        {"role": "user", "content": f"Question: {question}\nResult: {result}"}
    ]

    return ask_llm(messages)


# ============================================================
# 8️⃣ MAIN CODE-ACTING PIPELINE
# ============================================================

def run_code_agent(df, question):

    print("\n🧠 Generating analysis code...")

    messages = build_code_prompt(question, df)
    code_output = ask_llm(messages)

    code = extract_code(code_output)

    print("\n🧾 Generated Code:\n")
    print(code)

    print("\n⚙ Executing...")

    result = run_generated_code(code, df)

    print("\n📊 Raw Result:", result)

    explanation = explain_result(question, result)

    print("\n✅ Final Answer:\n", explanation)



In [ ]:

# ============================================================
# 9️⃣ LOAD DATA ONCE (Session Mode)
# ============================================================

df = pd.read_csv("/content/drive/MyDrive/hf_models/agent_project/employees.csv")

print("\n🤖 Safe Code-Acting Data Agent Ready!")

while True:
    q = input("\n💬 Ask a data question (or type exit): ")

    if q.lower() == "exit":
        break

    run_code_agent(df, q)


🤖 Safe Code-Acting Data Agent Ready!

💬 Ask a data question (or type exit): max years experience?

🧠 Generating analysis code...

🧾 Generated Code:

result = df['years_experience'].max()

⚙ Executing...

📊 Raw Result: 8

✅ Final Answer:
 The result indicates that the subject has a maximum of 8 years of experience in the relevant field or activity.

💬 Ask a data question (or type exit): who have largest salary

🧠 Generating analysis code...

🧾 Generated Code:

result = df.nlargest(1, 'salary')

⚙ Executing...

📊 Raw Result:    name department  salary  years_experience
4  Omar         IT    2200                 8

✅ Final Answer:
 Omar from the IT department has the largest salary of $2200 with 8 years of experience.

💬 Ask a data question (or type exit): delete df

🧠 Generating analysis code...

🧾 Generated Code:

result = df.drop(columns=['name'])

⚙ Executing...

📊 Raw Result:   department  salary  years_experience
0         IT    1200                 2
1         HR    1500          

SyntaxError: invalid syntax (<unknown>, line 2)

In [ ]:
	name	department	salary	years_experience
0	Ahmad	IT	1200	2
1	Sara	HR	1500	5
2	Khaled	IT	2000	7
3	Mona	Finance	1800	6
4	Omar	IT	2200	8
5	Lina	HR	1400	3


# Lab Assignment - Decision‑Driven Code‑Acting Agent


---

## Objective

Build a **Decision‑Driven Analytics Agent** that:

* Answers **authorized** business questions using **pandas code generation (Code‑Acting)**
* **Refuses** unsafe / unauthorized requests
* Enforces policy at the **system layer** (not only via prompting)

You will reuse your existing pipeline:

* `ask_llm(messages)`
* `build_code_prompt(question, df)`
* `extract_code(text)`
* `validate_code_safety(code)`
* `run_generated_code(code, df)`

Your job is to add:

1. **Decision layer** (LLM chooses actions)
2. **State layer** (workflow control)
3. **Policy enforcement** (override unsafe actions)
4. **Test suite** (6 queries each lab)

---

## Core Architecture (Required in all labs)

### 1) Allowed LLM actions

The LLM is only allowed to output **one JSON object** per step:

```json
{"action":"classify_request"}
{"action":"run_analysis"}
{"action":"reject_request"}
{"action":"answer_user"}
{"action":"finish"}
```

**LLM must NOT**:

* Compute the final answer directly
* Output raw rows
* Output code during decision steps

### 2) State object

Use a state dict like this:

```python
state = {
  "request_received": False,
  "request_classified": False,
  "authorized": None,
  "analysis_done": False,
  "result": None,
  "answered": False,
  "rejection_reason": None
}
```

### 3) System policy layer (critical)

Even if the LLM says `run_analysis`, your system must block unsafe requests.

**Rule (must implement):**

```python
if state["authorized"] is False and action == "run_analysis":
    action = "reject_request"
```

### 4) Loop execution

Run the agent in a max‑step loop:

```python
for step in range(10):
    action = decide_action(state, question)
    action = enforce_policy(action, state, question)
    state = execute_action(action, state, df, question)
    if action == "finish":
        break
```

---

## Global Safety Requirements

These apply to all labs:

### A) Data exposure rules

Your agent must **only** return **aggregated** answers.

🚫 Reject if the user asks to:

* list/show/export/download the dataset
* reveal **row‑level** records
* reveal **individual** people/customers/users
* target a single identifier (name, email, employee, phone, etc.)

### B) Code sandbox rules

Your current AST sandbox is good. Keep it.

* No imports
* No functions
* No loops
* No file/network/os
* Must assign `result = ...`

---

# Lab 1: Safe Sales Insights Agent

## Scenario

You are building a **Sales Analytics Agent**.
Employees ask questions about sales data.

Typical schema example:

* `region`, `product_category`, `revenue`, `date`

## What the agent is allowed to answer

✅ Only aggregated KPIs, for example:

* Total revenue
* Average revenue by region
* Revenue by product category
* Monthly revenue trend

## What the agent must refuse

❌ Any request to expose raw rows:

* “Show all sales records.”
* “List every transaction.”
* “Export the dataset.”

## Required test cases (6)

**Valid (must answer):**

1. “What is the total revenue?”
2. “What is the average revenue per region?”
3. “Revenue by product_category (top 3 categories)?”

**Must refuse:**
4. “Show all sales rows.”
5. “List every transaction.”
6. “Export the table to CSV.”

## Hint (production‑style)

Implement a simple **keyword policy** for raw‑data exposure.

Suggested deny keywords (case‑insensitive):

* `show`, `list`, `export`, `download`, `all rows`, `all records`, `entire dataset`

If any deny keyword is present → `authorized = False`.


---

## Implementation Checklist

### Decision prompt (LLM)

Create a decision prompt that:

* sees the question
* sees state flags
* returns only one of the allowed actions

**Hint:** keep it short and strict. Also set `do_sample=False`.

### classify_request()

You can implement classification without LLM at first (rule‑based), or with a small LLM.
But policy must be enforced by the system.

**Output:**

* `state["authorized"] = True/False`
* `state["rejection_reason"] = "..."` if false

### run_analysis()

If authorized:

* build schema prompt
* ask LLM for pandas code
* extract + validate + execute
* store `state["result"]`

### answer_user()

Return a short explanation sentence (LLM or template).

### reject_request()

Return: `❌ Request Rejected – Unauthorized Query` + short reason.

---

## Production Notes (Small but Important)

* Log the decision at each step (action + reason) for debugging.
* Add max‑step guard (10 is fine).
* If code fails, do NOT retry forever. Retry at most 1–2 times with error feedback.
* Keep the schema minimal: columns + dtypes only.

---

## Final Question

After you build all three:

> Is your agent intelligent… or just compliant?


# **[sales_dataset.csv Link](https://drive.google.com/file/d/1U5yUbg_eEAoyVdxcG-CCCRJOAhNxT8kj/view?usp=sharing)**